# Exploratory Data Analysis - POS / Cash Balance Dataset

EDA on `POS_CASH_balance.csv` from the Home Credit Default Risk competition: monthly snapshots of a client's previous Home Credit **POS (point of sale) and cash** loans. This is a panel table — one row per `(SK_ID_PREV, MONTHS_BALANCE)`, so each previous credit contributes many monthly rows.

**Related Notebooks:**
- `application_eda.ipynb` - Application_train dataset
- `bureau_eda.ipynb` - Bureau dataset
- `previous_application_eda.ipynb` - Previous applications
- `credit_card_balance_eda.ipynb` - Credit card balances
- `installments_payments_eda.ipynb` - Installment payments

**Analysis Sections:**
1. Setup and Data Loading
2. Basic Dataset Exploration
3. Missing Data Analysis
4. Numeric Features Analysis
5. Categorical Features Analysis
6. Standard Data Quality Checks
7. Functional / Business-Logic Quality Checks

## 1. Setup and Data Loading

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

# Add src to path to import custom modules
sys.path.insert(0, "/Users/bharathregula/code/home_credit_risk_deeplearn/src")
from home_credit_risk.eda_utils import (
    analyze_categorical_features,
    analyze_numeric_features,
    check_categorical_placeholders,
    check_data_quality,
    check_duplicates,
    detect_sentinel_values,
    display_basic_info,
    show_missing_patterns,
)

# Configure polars display options
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)  # show all columns
pl.Config.set_fmt_str_lengths(80)

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Load dataset
datasets = Path("/Users/bharathregula/code/datasets/home-credit-default-risk")

# Every check below reuses the whole frame, so we collect it once. A large
# infer_schema_length guards against dtype mis-inference across the columns.
pos_df = pl.read_csv(datasets / "POS_CASH_balance.csv", infer_schema_length=100000)

print("Dataset loaded successfully!")
print(f"Shape: {pos_df.shape}")

In [ ]:
# Quick preview of the data
pos_df.head()

## 2. Basic Dataset Exploration

In [ ]:
# Display basic dataset information
display_basic_info(pos_df, df_name="POS_CASH_Balance")

In [ ]:
# Summary statistics
pos_df.describe()

## 3. Missing Data Analysis

This table has a **composite** primary key `(SK_ID_PREV, MONTHS_BALANCE)` — a single previous credit spans many months — so full-row duplicates are checked here and the composite-key uniqueness is asserted in the functional section.

In [ ]:
# Analyze missing data patterns
missing_summary = show_missing_patterns(pos_df, threshold=0.0)

In [ ]:
# Check for full-row duplicate records
check_duplicates(pos_df)

## 4. Numeric Features Analysis

In [ ]:
# Analyze numeric features
numeric_df = analyze_numeric_features(pos_df, figsize=(18, 6), show_summary=True)

## 5. Categorical Features Analysis

In [ ]:
# Analyze categorical features
analyze_categorical_features(pos_df)

## 6. Standard Data Quality Checks

In [ ]:
# Generic data quality checks
check_data_quality(pos_df)

## 7. Functional / Business-Logic Quality Checks

Cross-field consistency rules for a monthly POS/cash-loan panel: installment bounds (you cannot owe more than the term), the days-past-due ordering, and status ↔ remaining-installments consistency.

In [ ]:
# Small helper for consistent count + percentage reporting
n = pos_df.height


def pct(count: int) -> str:
    return f"{count:,} ({count / n * 100:.3f}%)"


print(f"Total monthly POS/cash rows: {n:,}")

### A. Grain / key integrity

Each `(SK_ID_PREV, MONTHS_BALANCE)` pair should be unique, and `MONTHS_BALANCE` is relative to the application date so it must be ≤ 0.

In [ ]:
# A1. Composite-key uniqueness
grain_dupes = n - pos_df.select("SK_ID_PREV", "MONTHS_BALANCE").n_unique()
print(f"A1. Duplicate (SK_ID_PREV, MONTHS_BALANCE) rows: {pct(grain_dupes)}")

# A2. MONTHS_BALANCE must be <= 0 (past months relative to application)
a2 = pos_df.filter(pl.col("MONTHS_BALANCE") > 0).height
print(f"A2. MONTHS_BALANCE > 0 (future months): {pct(a2)}")

### B. Installment bounds

Installments left to pay cannot exceed the (current) term of the credit: `CNT_INSTALMENT_FUTURE ≤ CNT_INSTALMENT`. Only rows where both are non-null are checked.

In [ ]:
# B1. CNT_INSTALMENT_FUTURE <= CNT_INSTALMENT
b1_sub = pos_df.filter(
    pl.col("CNT_INSTALMENT").is_not_null()
    & pl.col("CNT_INSTALMENT_FUTURE").is_not_null()
)
b1 = b1_sub.filter(pl.col("CNT_INSTALMENT_FUTURE") > pl.col("CNT_INSTALMENT"))
print(
    f"B1. CNT_INSTALMENT_FUTURE > CNT_INSTALMENT (of {b1_sub.height:,} checked): "
    f"{pct(b1.height)}"
)
b1.select(
    "SK_ID_PREV", "MONTHS_BALANCE", "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"
).head(10)

### C. Days-past-due relationship

`SK_DPD_DEF` is DPD with a tolerance that ignores small debts, so it can only be ≤ the raw `SK_DPD`.

In [ ]:
# C1. SK_DPD_DEF <= SK_DPD
c1 = pos_df.filter(pl.col("SK_DPD_DEF") > pl.col("SK_DPD")).height
print(f"C1. SK_DPD_DEF > SK_DPD: {pct(c1)}")

### D. Non-negativity

Installment counts and days-past-due should never be negative.

In [ ]:
# D1. Non-negativity
nonneg_cols = [
    "CNT_INSTALMENT",
    "CNT_INSTALMENT_FUTURE",
    "SK_DPD",
    "SK_DPD_DEF",
]
for col in nonneg_cols:
    neg = pos_df.filter(pl.col(col) < 0).height
    print(f"D1. {col} < 0: {pct(neg)}")

### E. Status ↔ remaining-installments consistency

A `Completed` contract should have nothing left to pay (`CNT_INSTALMENT_FUTURE == 0`). An `Active` contract with zero installments left is reported as *informational* — it is typically a contract about to flip to completed.

In [ ]:
# E1. Completed contracts with installments still outstanding
completed = pos_df.filter(pl.col("NAME_CONTRACT_STATUS") == "Completed")
e1 = completed.filter(pl.col("CNT_INSTALMENT_FUTURE") > 0)
print(
    f"E1. Completed rows with CNT_INSTALMENT_FUTURE > 0: {e1.height} "
    f"of {completed.height:,} completed"
)

# E2. Active contracts already fully paid down (informational)
active = pos_df.filter(pl.col("NAME_CONTRACT_STATUS") == "Active")
e2 = active.filter(pl.col("CNT_INSTALMENT_FUTURE") == 0)
print(
    f"E2. Active rows with CNT_INSTALMENT_FUTURE == 0 (informational): {e2.height} "
    f"of {active.height:,} active"
)

### F. Sentinel / placeholder detection

Reuse the shared helpers. `detect_sentinel_values` auto-skips `SK_ID*` identifier columns; `check_categorical_placeholders` surfaces `XNA`/`XAP` tokens in `NAME_CONTRACT_STATUS`.

In [ ]:
# F1. Numeric sentinel + categorical placeholders
sentinel_summary = detect_sentinel_values(pos_df)
placeholder_summary = check_categorical_placeholders(pos_df)